# 03 Evaluation (Standalone)

This notebook loads a saved checkpoint and evaluates the official CUB-200 test set.
It reports Top-1 accuracy and average accuracy per class, then visualizes
overall metrics and the strongest/weakest classes.


In [ ]:
# Import evaluation, plotting, and PyTorch utilities.
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
NUM_CLASSES=200; IMAGE_SIZE=224; BATCH_SIZE=32; NW=2
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
ROOT=Path('..').resolve(); DATA=ROOT/'data'; CK=ROOT/'models'/'checkpoints'; RS=ROOT/'models'/'results'


In [ ]:
# Use deterministic preprocessing for the official test images.
class CUB(Dataset):
 def __init__(self,idir,at,tf=None):
  self.idir=Path(idir); self.tf=tf; self.samples=[]
  with open(at,'r',encoding='utf-8') as f:
   for l in f:
    l=l.strip()
    if not l: continue
    p,y=l.split(); self.samples.append((self.idir/p,int(y)))
 def __len__(self): return len(self.samples)
 def __getitem__(self,i):
  p,y=self.samples[i]; im=Image.open(p).convert('RGB'); return (self.tf(im) if self.tf else im),y
def tfm():
 return transforms.Compose([transforms.Resize(256),transforms.CenterCrop(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
def test_loader(data_dir='data',bs=32,nw=2):
 d=Path(data_dir); ds=CUB(d/'Test',d/'test.txt',tfm())
 return DataLoader(ds,bs,False,num_workers=nw,pin_memory=torch.cuda.is_available())
def model_of(name):
 # The architecture must match the architecture used to create the checkpoint.
 n=name.lower()
 if n=='resnet18': m=models.resnet18(weights=None); m.fc=nn.Linear(m.fc.in_features,NUM_CLASSES)
 elif n=='resnet50': m=models.resnet50(weights=None); m.fc=nn.Linear(m.fc.in_features,NUM_CLASSES)
 elif n=='efficientnet_b0': m=models.efficientnet_b0(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
 else: raise ValueError('unsupported')
 return m
def load_best(name,device):
 p=CK/f'{name}_best.pth'
 if not p.exists(): raise FileNotFoundError(p)
 # Load only the saved model weights; pretrained download is not needed here.
 m=model_of(name); ck=torch.load(p,map_location=device); m.load_state_dict(ck['model_state_dict']); m.to(device).eval(); return m,ck,p


In [ ]:
# Count correct predictions globally and separately for every class.
@torch.no_grad()
def eval_model(m,ldr,device):
 cor=torch.zeros(NUM_CLASSES,dtype=torch.long); tot=torch.zeros(NUM_CLASSES,dtype=torch.long)
 for x,y in tqdm(ldr,desc='test',leave=False,dynamic_ncols=True):
  x=x.to(device,non_blocking=True); yc=y.cpu(); pred=m(x).argmax(1).cpu()
  tot+=torch.bincount(yc,minlength=NUM_CLASSES); mask=(pred==yc); cor+=torch.bincount(yc[mask],minlength=NUM_CLASSES)
 prs=(tot>0); idx=np.where(prs.numpy())[0]; per=(cor[prs].float()/tot[prs].float()).numpy(); top1=float(cor.sum().item()/tot.sum().item()); avg=float(per.mean())
 return top1,avg,idx,per
def plot_eval(top1,avg,idx,per):
 # These charts make class imbalance and difficult classes easy to inspect.
 p=per*100; fig,ax=plt.subplots(2,2,figsize=(14,10))
 ax[0,0].bar(['Top1','Avg/Class'],[top1*100,avg*100]); ax[0,0].set_ylim(0,100); ax[0,0].set_title('core metrics')
 ax[0,1].hist(p,bins=20,color='green'); ax[0,1].set_title('per-class distribution')
 o=np.argsort(p); b=o[:10]; t=o[-10:]
 ax[1,0].bar(range(10),p[b],color='red'); ax[1,0].set_xticks(range(10)); ax[1,0].set_xticklabels(idx[b],rotation=45); ax[1,0].set_title('bottom-10')
 ax[1,1].bar(range(10),p[t],color='purple'); ax[1,1].set_xticks(range(10)); ax[1,1].set_xticklabels(idx[t],rotation=45); ax[1,1].set_title('top-10')
 plt.tight_layout(); plt.show()


In [ ]:
# Evaluate the selected checkpoint and save the required assignment metrics.
MODEL='resnet50'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ldr=test_loader(DATA,BATCH_SIZE,NW); m,ck,cp=load_best(MODEL,device)
top1,avg,idx,per=eval_model(m,ldr,device); plot_eval(top1,avg,idx,per)
RS.mkdir(parents=True,exist_ok=True); rp=RS/f'{MODEL}_test_results.json'
with open(rp,'w',encoding='utf-8') as f: json.dump({'model_name':MODEL,'best_validation_accuracy':float(ck['validation_accuracy']),'test_top1_accuracy':float(top1),'test_average_per_class_accuracy':float(avg),'test_samples':len(ldr.dataset),'classes_evaluated':int(len(idx))},f,indent=4)
print('checkpoint:',cp,'\nresult:',rp,'\nTop1:',f'{top1*100:.2f}%','\nAvg/Class:',f'{avg*100:.2f}%')
